# Approach 2: Pure Transformer with Manual Feature Extraction (NLP Paradigm)

This notebook implements the NLP-style approach to EEG translation. Instead of feeding raw waveform amplitudes to the network, we manually engineer features by breaking the brain waves into frequency bands (Alpha, Beta, Theta, Gamma) using SciPy, treating these extracted bands as "tokens" similar to grammatical features in a sentence.

In [ ]:
import difflib

import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm


def greedy_decoder(logits, char_to_idx):
    idx_to_char = {v: k for k, v in char_to_idx.items()}
    preds = torch.argmax(logits, dim=-1).transpose(0, 1)
    decoded = []
    for seq in preds:
        prev = -1
        text = ""
        for c in seq:
            c = c.item()
            if c != 0 and c != prev:
                text += idx_to_char.get(c, "")
            prev = c
        decoded.append(text)
    return decoded


def calculate_cer(pred_texts, target_texts):
    scores = []
    for p, t in zip(pred_texts, target_texts):
        scores.append(difflib.SequenceMatcher(None, p, t).ratio())
    return sum(scores) / max(1, len(scores))

In [ ]:
# Requirements: pip install torch scipy h5py matplotlib
import glob
import os

import h5py
import numpy as np
import torch
from scipy.signal import butter, lfilter
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset

DATASET_PATH = r"C:\SDE Projects\Open-BCI-EEG-Waves-To-Text-Translation-And-further-Robotic-Implementaions\dataset\extracted"
MODELS_DIR = "models"
METRICS_DIR = "metrics"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

## 1. Feature Extraction (Frequency Bands)
Extracting specific frequency ranges: Theta (4-8 Hz), Alpha (8-13 Hz), Beta (13-30 Hz), Gamma (30-100 Hz).

In [ ]:
def butter_bandpass(lowcut, highcut, fs, order=3):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype="band")
    return b, a


def extract_frequency_bands(data, fs=500):
    # data shape: [105, time_steps]
    bands = {"theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 100)}
    features = []
    for low, high in bands.values():
        b, a = butter_bandpass(low, high, fs)
        filtered = lfilter(b, a, data, axis=1)
        # Take signal envelope or power (squaring for simple power estimate)
        power = np.square(filtered)
        features.append(power)
    # Stack into [4_bands * 105_channels, time_steps]
    return np.vstack(features)


class EEGFeatureDataset(Dataset):
    def __init__(self, data_dir, max_len=500):
        self.files = glob.glob(os.path.join(data_dir, "**", "*.h5"), recursive=True)
        self.max_len = max_len
        self.chars = (
            "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,!?'-"
        )
        self.char_to_idx = {ch: i + 1 for i, ch in enumerate(self.chars)}
        self.char_to_idx["<PAD>"] = 0
        self.vocab_size = len(self.char_to_idx)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        filename = os.path.basename(file_path)
        transcript = filename.replace(".h5", "").replace("_", " ")

        with h5py.File(file_path, "r") as f:
            keys = list(f.keys())
            if len(keys) > 0:
                eeg_data = f[keys[0]][:]
            else:
                eeg_data = np.zeros((105, self.max_len))

        if len(eeg_data.shape) != 2 or eeg_data.shape[0] != 105:
            eeg_data = np.zeros((105, self.max_len))
        if eeg_data.shape[1] < self.max_len:
            pad_width = self.max_len - eeg_data.shape[1]
            eeg_data = np.pad(eeg_data, ((0, 0), (0, pad_width)), mode="constant")
        else:
            eeg_data = eeg_data[:, : self.max_len]

        # Extract NLP-like features (Bands)
        features = extract_frequency_bands(eeg_data)
        # Features shape will be [420, max_len]
        feature_tensor = torch.tensor(features, dtype=torch.float32).transpose(0, 1)

        target = [self.char_to_idx.get(c, 0) for c in transcript]
        target_tensor = torch.tensor(target, dtype=torch.long)
        return feature_tensor, target_tensor


dataset = EEGFeatureDataset(DATASET_PATH)
print(f"Found {len(dataset)} HDF5 files for training.")

## 2. Pure Transformer Architecture
Using a standard NLP-style Transformer on the engineered features.

In [ ]:
class PureTransformer(nn.Module):
    def __init__(
        self, feature_dim=420, d_model=256, nhead=8, num_layers=4, num_classes=65
    ):
        super().__init__()
        self.embedding = nn.Linear(feature_dim, d_model)
        self.pos_encoder = nn.Parameter(torch.randn(1, 1000, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        # x shape: [batch, time, feature_dim]
        x = self.embedding(x)
        x = x + self.pos_encoder[:, : x.size(1), :]
        out = self.transformer(x)
        out = self.fc(out)
        return out

## 3. Training Loop

In [ ]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

if len(dataset) > 0:
    model = PureTransformer(num_classes=dataset.vocab_size)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    epochs = 3
    loss_history = []
    recall_history = []
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    display_handle = display(fig, display_id=True)
    plt.close(fig)
    model.train()
    print("Training Pure Transformer (NLP Paradigm)...")
    for epoch in range(epochs):
        epoch_loss = 0
        epoch_recall = 0
        pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{epochs}")
        for feat, target, subject in pbar:
            batch_size = feat.size(0)
            input_lengths = torch.full(
                size=(batch_size,), fill_value=feat.size(1), dtype=torch.long
            )
            target_lengths = torch.tensor([len(target[0])])

            optimizer.zero_grad()
            out = model(feat)
            out = out.transpose(0, 1)
            out = nn.functional.log_softmax(out, dim=2)

            loss = criterion(out, target, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

            batch_recall = 0
            if "char_to_idx" in globals() or hasattr(dataset, "char_to_idx"):
                char_to_idx = (
                    dataset.char_to_idx
                    if hasattr(dataset, "char_to_idx")
                    else globals()["char_to_idx"]
                )
                preds = greedy_decoder(out, char_to_idx)
                idx_to_char = {v: k for k, v in char_to_idx.items()}
                targets_text = []
                for t in target:
                    t_list = t if isinstance(t, list) else t[0].tolist()
                    targets_text.append(
                        "".join([idx_to_char.get(c, "") for c in t_list])
                    )
                batch_recall = calculate_cer(preds, targets_text)
            epoch_recall += batch_recall

            ax1.clear()
            ax2.clear()
            ax3.clear()
            subj = subject[0] if len(subject) > 0 else "Unknown"

            sample = feat[0].detach().cpu().numpy()[:50, :]
            ax1.plot(sample)
            ax1.legend(["Gamma", "Beta", "Alpha", "Theta", "Delta"], loc="upper right")
            ax1.set_title(f"Live NLP Bands (Subj: {subj})")
            ax1.set_xlabel("Window")

            ax2.plot(loss_history + [loss.item()], color="r", label="CTC Loss")
            ax2.set_title("Real-Time CTC Loss")
            ax2.set_xlabel("Batches")
            ax2.legend(loc="upper right")

            ax3.plot(
                recall_history + [batch_recall], color="g", label="Recall (Accuracy)"
            )
            ax3.set_title("Real-Time Character Recall")
            ax3.set_xlabel("Batches")
            ax3.legend(loc="upper right")

            display_handle.update(fig)
            pbar.set_postfix(
                {"Loss": f"{loss.item():.4f}", "Recall": f"{batch_recall:.4f}"}
            )

        avg_loss = epoch_loss / max(1, len(dataloader))
        avg_recall = epoch_recall / max(1, len(dataloader))
        recall_history.append(avg_recall)
        loss_history.append(avg_loss)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.4f}")

    torch.save(
        model.state_dict(), os.path.join(MODELS_DIR, "nlp_transformer_checkpoint.pth")
    )

    plt.plot(loss_history, marker="s")
    plt.title("NLP Paradigm Transformer Loss")
    plt.xlabel("Epoch")
    plt.ylabel("CTC Loss")
    plt.savefig(os.path.join(METRICS_DIR, "nlp_loss_curve.png"))
    print("Training Complete!")
else:
    print("No HDF5 data found.")